In [4]:
import requests
import pandas as pd

BASE = "https://api.openf1.org/v1"

def get(endpoint, **params):
    r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json())

# 1) Yarışın session_key değerini bul
sessions = get("sessions", year=2024, country_name="Bahrain", session_name="Race")
print(sessions[["session_key", "session_name", "country_name", "date_start"]])
session_key = int(sessions.loc[0, "session_key"])

# 2) Turlar ve lastik stintleri
laps = get("laps", session_key=session_key)
stints = get("stints", session_key=session_key)

print("Tur sayısı:", len(laps), "| Stint sayısı:", len(stints))
laps.head()

   session_key session_name country_name                 date_start
0         9472         Race      Bahrain  2024-03-02T15:00:00+00:00
Tur sayısı: 1129 | Stint sayısı: 63


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed
0,1229,9472,1,1,2024-03-02T15:03:42.341000+00:00,32.877,41.266,23.616,234.0,250,False,97.759,"[None, 2048, 2049, 2049, 2051, 2051, 2049, 204...","[2051, 2051, 2051, 2049, 2049, 2049, 2049, 205...","[2051, 2051, 2049, 2049, 2051, 2049, 2051]",251.0
1,1229,9472,16,1,2024-03-02T15:03:42.341000+00:00,33.495,41.549,23.635,232.0,252,False,98.679,"[None, 2048, 2049, 2051, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2051, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2051]",255.0
2,1229,9472,63,1,2024-03-02T15:03:42.341000+00:00,33.889,42.099,23.694,225.0,252,False,99.682,"[None, 2048, 2049, 2049, 2049, 2049, 2051, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2051]",256.0
3,1229,9472,44,1,2024-03-02T15:03:42.341000+00:00,35.450,43.609,24.393,229.0,246,False,103.452,"[2048, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2051, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049]",249.0
4,1229,9472,55,1,2024-03-02T15:03:42.341000+00:00,34.408,42.861,23.765,233.0,249,False,101.034,"[None, 2048, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2051, 2049, 2051, 2051]",256.0


In [5]:
df = laps.merge(stints, on="driver_number", suffixes=("", "_stint"))
df = df[(df.lap_number >= df.lap_start) & (df.lap_number <= df.lap_end)].copy()
df["tyre_life"] = df.tyre_age_at_start + (df.lap_number - df.lap_start)

df[["driver_number", "lap_number", "lap_duration", "compound", "tyre_life"]].head(20)

,driver_number,lap_number,lap_duration,compound,tyre_life
0,1,1,97.759,SOFT,3
3,16,1,98.679,SOFT,3
6,63,1,99.682,SOFT,3
9,44,1,103.452,SOFT,3
12,55,1,101.034,SOFT,3
15,11,1,100.537,SOFT,3
18,27,1,113.929,SOFT,0
22,23,1,104.246,SOFT,0
25,14,1,102.058,SOFT,3
28,77,1,108.853,SOFT,0


In [7]:
import os
os.makedirs(os.path.expanduser("~/f1-strategy/data/raw"), exist_ok=True)
df.to_csv(os.path.expanduser("~/f1-strategy/data/raw/bahrain_2024_race.csv"), index=False)